# MDM2 amplification and the immune microenvironment in sarcoma
Public data only (cBioPortal, GEO). In Colab: Runtime > Run all. Section 6 downloads ~650 MB.

## 1. Setup

In [1]:
import os
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)
print("ready")

ready


## 2. Cox proportional hazards helper

In [2]:
"""Minimal Cox proportional hazards (Breslow ties) and helpers, numpy/scipy only."""
import numpy as np
import pandas as pd
from scipy import stats


def cox(df, time_col, event_col, covariates, max_iter=50, tol=1e-8):
    """Fit a Cox PH model by Newton-Raphson. Returns a DataFrame with HR, 95% CI and Wald p."""
    d = df[[time_col, event_col] + covariates].dropna()
    d = d[d[time_col] > 0]
    X = d[covariates].astype(float).values
    keep = X.std(axis=0) > 0
    X, names = X[:, keep], [c for c, k in zip(covariates, keep) if k]
    t, e = d[time_col].values.astype(float), d[event_col].values.astype(float)
    order = np.argsort(-t)
    t, e, X = t[order], e[order], X[order]
    last = np.searchsorted(-t, -t, side="right") - 1      # last index of each tied risk set

    def grad_hess(b):
        r = np.exp(X @ b)
        cs = np.cumsum(r)
        csx = np.cumsum(r[:, None] * X, axis=0)
        csxx = np.cumsum(r[:, None, None] * X[:, :, None] * X[:, None, :], axis=0)
        g = np.zeros(X.shape[1])
        H = np.zeros((X.shape[1],) * 2)
        for i in np.where(e == 1)[0]:
            j = last[i]
            m = csx[j] / cs[j]
            g += X[i] - m
            H -= csxx[j] / cs[j] - np.outer(m, m)
        return g, H

    b = np.zeros(X.shape[1])
    for _ in range(max_iter):
        g, H = grad_hess(b)
        step = np.linalg.solve(H, g)
        b = b - step
        if np.max(np.abs(step)) < tol:
            break
    _, H = grad_hess(b)
    se = np.sqrt(np.diag(np.linalg.inv(-H)))
    p = 2 * stats.norm.sf(np.abs(b / se))
    out = pd.DataFrame({"HR": np.exp(b), "CI_low": np.exp(b - 1.96 * se),
                        "CI_high": np.exp(b + 1.96 * se), "p": p}, index=names)
    out.attrs.update(n=len(d), events=int(e.sum()))
    return out


def fmt(res):
    rows = [f"{i}: HR {r.HR:.2f} ({r.CI_low:.2f}-{r.CI_high:.2f}), p={r.p:.3g}" for i, r in res.iterrows()]
    return "; ".join(rows) + f"  [n={res.attrs['n']}, events={res.attrs['events']}]"


def dummies(series, ref=None, min_n=8):
    """One-hot encode a categorical, dropping the reference level."""
    d = pd.get_dummies(series, prefix="h", dtype=float)
    if ref is not None and f"h_{ref}" in d:
        d = d.drop(columns=f"h_{ref}")
    else:
        d = d.iloc[:, 1:]
    return d


cox_base = cox  # kept separately so later cells can wrap it safely


## 3. Download TCGA-SARC and MSK-IMPACT 50K from cBioPortal

In [3]:
# re-download if data are missing or came from the older (paged) downloader
if open("data/VERSION").read().strip() == "v2" if os.path.exists("data/VERSION") else False:
    print("cBioPortal data already downloaded")
else:
    """Download the public data used in this project from the cBioPortal REST API.

    Outputs (data/):
      tcga_sarc.json       TCGA-SARC PanCancer Atlas: clinical, GISTIC CNA, RNA-seq for selected genes
      msk50k_sarcoma.json  MSK-IMPACT 50K: sarcoma samples, TMB and GISTIC CNA for MDM2/MDM4/CDK4
    """
    import json, os, requests

    API = "https://www.cbioportal.org/api"
    os.makedirs("data", exist_ok=True)
    S = requests.Session()


    import time


    def _call(method, path, body=None, params=None, tries=6):
        for k in range(tries):
            try:
                r = S.request(method, f"{API}{path}", json=body, params=params, timeout=300)
                if r.status_code < 500:
                    r.raise_for_status()
                    return r.json()
                print(f"  cBioPortal {r.status_code} on {path}, retrying ({k + 1}/{tries})")
            except requests.exceptions.RequestException as e:
                if k == tries - 1:
                    raise
                print(f"  network error on {path}: {e}; retrying ({k + 1}/{tries})")
            time.sleep(5 * (k + 1))
        raise RuntimeError(f"cBioPortal kept failing on {path}")


    def get(path, **params):
        """GET with retries (no paging: cBioPortal pages are not stably ordered)."""
        return _call("GET", path, params=params)


    def post(path, body, **params):
        return _call("POST", path, body=body, params=params)


    # ---------------- TCGA-SARC ----------------
    study = "sarc_tcga_pan_can_atlas_2018"
    GENES = ["FOXP3", "IL2RA", "CTLA4", "IKZF2", "TNFRSF18", "CCR8", "TNFRSF4",
             "CD8A", "CD8B", "GZMA", "GZMB", "GZMK", "PRF1", "NKG7", "GNLY",
             "CD68", "CD163", "CSF1R", "C1QA", "C1QB", "MRC1",
             "MDM2", "MDM4", "CDK4", "CD274", "PTPRC", "CD3E", "IFNG", "CXCL9", "CXCL10", "HMGA2", "FRS2"]
    genes = post("/genes/fetch", GENES, geneIdType="HUGO_GENE_SYMBOL")
    sym = {g["entrezGeneId"]: g["hugoGeneSymbol"] for g in genes}
    ids = list(sym)
    cna_ids = [i for i in ids if sym[i] in ("MDM2", "MDM4", "CDK4", "HMGA2", "FRS2")]

    expr = post(f"/molecular-profiles/{study}_rna_seq_v2_mrna/molecular-data/fetch",
                {"sampleListId": f"{study}_all", "entrezGeneIds": ids}, projection="SUMMARY")
    cna = post(f"/molecular-profiles/{study}_gistic/molecular-data/fetch",
               {"sampleListId": f"{study}_all", "entrezGeneIds": cna_ids}, projection="SUMMARY")

    KEEP_P = {"AGE", "SEX", "OS_MONTHS", "OS_STATUS", "DSS_MONTHS", "DSS_STATUS", "PFS_MONTHS", "PFS_STATUS",
              "SUBTYPE", "AJCC_PATHOLOGIC_TUMOR_STAGE", "ICD_O_3_HISTOLOGY", "RADIATION_THERAPY", "HISTORY_NEOADJUVANT_TRTYN"}
    KEEP_S = {"CANCER_TYPE_DETAILED", "ONCOTREE_CODE", "TMB_NONSYNONYMOUS", "ANEUPLOIDY_SCORE",
              "FRACTION_GENOME_ALTERED", "TUMOR_TISSUE_SITE", "SAMPLE_TYPE", "MUTATION_COUNT"}
    pc, sc = {}, {}
    for a in sorted(KEEP_P):   # one small request per attribute
        for x in get(f"/studies/{study}/clinical-data", clinicalDataType="PATIENT", attributeId=a, projection="SUMMARY", pageSize=100000):
            pc[(x["patientId"], a)] = x["value"]
    for a in sorted(KEEP_S):
        for x in get(f"/studies/{study}/clinical-data", clinicalDataType="SAMPLE", attributeId=a, projection="SUMMARY", pageSize=100000):
            sc[(x["sampleId"], a)] = (x["patientId"], x["value"])
    tcga = {
        "pc": [[pid, a, v] for (pid, a), v in pc.items()],
        "sc": [[sid, pid, a, v] for (sid, a), (pid, v) in sc.items()],
        "expr": [[x["sampleId"], sym[x["entrezGeneId"]], x["value"]] for x in expr],
        "cna": [[x["sampleId"], sym[x["entrezGeneId"]], x["value"]] for x in cna],
    }
    json.dump(tcga, open("data/tcga_sarc.json", "w"))
    print("TCGA-SARC:", {k: len(v) for k, v in tcga.items()})

    # ---------------- MSK-IMPACT 50K ----------------
    study = "msk_impact_50k_2026"
    TYPES = {"Uterine Sarcoma", "Soft Tissue Sarcoma", "Bone Cancer", "Breast Sarcoma", "Soft Tissue Cancer",
             "Clear Cell Sarcoma of Kidney", "Undifferentiated Embryonal Sarcoma of the Liver"}
    ct = get(f"/studies/{study}/clinical-data", clinicalDataType="SAMPLE", attributeId="CANCER_TYPE", pageSize=100000)
    samp = {x["sampleId"]: {"patient": x["patientId"], "cancer_type": x["value"]} for x in ct if x["value"] in TYPES}
    for attr, key in [("CANCER_TYPE_DETAILED", "detailed"), ("TMB_SCORE", "tmb")]:
        for x in get(f"/studies/{study}/clinical-data", clinicalDataType="SAMPLE", attributeId=attr, pageSize=100000):
            if x["sampleId"] in samp:
                samp[x["sampleId"]][key] = x["value"]
    cna_profiled = set(get(f"/sample-lists/{study}_cna/sample-ids"))
    ids = [s for s in samp if s in cna_profiled]
    GMAP = {4193: "MDM2", 4194: "MDM4", 1019: "CDK4"}
    for i in range(0, len(ids), 1000):
        for x in post(f"/molecular-profiles/{study}_gistic/molecular-data/fetch",
                      {"sampleIds": ids[i:i + 1000], "entrezGeneIds": list(GMAP)}, projection="SUMMARY"):
            samp[x["sampleId"]][GMAP[x["entrezGeneId"]]] = x["value"]
    msk = [dict(sample=s, **samp[s]) for s in ids]
    json.dump(msk, open("data/msk50k_sarcoma.json", "w"))
    print("MSK-IMPACT sarcoma samples with CNA:", len(msk))

    open("data/VERSION", "w").write("v2")


TCGA-SARC: {'pc': 3018, 'sc': 2025, 'expr': 8096, 'cna': 1265}
MSK-IMPACT sarcoma samples with CNA: 3293


## 4. TCGA-SARC: immune microenvironment and survival

In [4]:
"""TCGA-SARC: MDM2/MDM4 amplification, immune microenvironment and survival (histology-adjusted)."""
import json, os, sys, numpy as np, pandas as pd
from scipy import stats
_cox = cox_base  # from the survival section

raw = json.load(open('data/tcga_sarc.json'))
pc = pd.DataFrame(raw['pc'], columns=['patient', 'attr', 'val']).pivot_table(index='patient', columns='attr', values='val', aggfunc='first')
sc = pd.DataFrame(raw['sc'], columns=['sample', 'patient', 'attr', 'val']).pivot_table(index=['sample', 'patient'], columns='attr', values='val', aggfunc='first').reset_index()
ex = pd.DataFrame(raw['expr'], columns=['sample', 'gene', 'v']).pivot_table(index='sample', columns='gene', values='v')
cn = pd.DataFrame(raw['cna'], columns=['sample', 'gene', 'v']).pivot_table(index='sample', columns='gene', values='v')

# primary tumors, one per patient
sc = sc[sc['SAMPLE_TYPE'].fillna('Primary').str.contains('Primary')].drop_duplicates('patient')
df = sc.merge(pc, left_on='patient', right_index=True, how='left').set_index('sample')
df = df.join(cn.add_prefix('cna_'), how='left')

lx = np.log2(ex.clip(lower=0) + 1)
z = (lx - lx.mean()) / lx.std()
sig = {
    'Treg': ['FOXP3', 'IL2RA', 'CTLA4', 'IKZF2', 'TNFRSF18', 'CCR8', 'TNFRSF4'],
    'Cytotoxic': ['CD8A', 'CD8B', 'GZMA', 'GZMB', 'GZMK', 'PRF1', 'NKG7', 'GNLY'],
    'Macrophage': ['CD68', 'CD163', 'CSF1R', 'C1QA', 'C1QB', 'MRC1'],
    'Tcell': ['CD3E'],
    'IFNG': ['IFNG', 'CXCL9', 'CXCL10'],
}
S = pd.DataFrame({k: z[v].mean(axis=1) for k, v in sig.items()})
S['Cyto_minus_Treg'] = S['Cytotoxic'] - S['Treg']          # log-scale cytotoxic:Treg balance
S['Treg_vs_Tcell'] = S['Treg'] - S['Tcell']                 # Treg enrichment relative to total T cells
S['MDM2_expr'] = z['MDM2']
df = df.join(S, how='left')

df['amp'] = ((df['cna_MDM2'] == 2) | (df['cna_MDM4'] == 2)).astype(float)
df.loc[df['cna_MDM2'].isna() & df['cna_MDM4'].isna(), 'amp'] = np.nan
df['cdk4_amp'] = (df['cna_CDK4'] == 2).astype(float)
df.loc[df['cna_CDK4'].isna(), 'cdk4_amp'] = np.nan
df['hist'] = df['CANCER_TYPE_DETAILED']
df['age'] = pd.to_numeric(df['AGE'], errors='coerce')
df['TMB'] = pd.to_numeric(df['TMB_NONSYNONYMOUS'], errors='coerce')
for e in ['OS', 'DSS', 'PFS']:
    df[e + '_t'] = pd.to_numeric(df[e + '_MONTHS'], errors='coerce')
    df[e + '_e'] = df[e + '_STATUS'].astype(str).str.startswith('1').astype(float)
    df.loc[df[e + '_STATUS'].isna(), e + '_e'] = np.nan

def cox(d, tcol, ecol, covs):
    r = _cox(d, tcol, ecol, covs).rename(columns={'CI_low': 'lo', 'CI_high': 'hi'})
    return r

def hist_dummies(d, ref='Leiomyosarcoma'):
    dd = pd.get_dummies(d['hist'], prefix='h', dtype=float)
    refc = 'h_' + ref
    if refc in dd: dd = dd.drop(columns=refc)
    return dd

def run(d, tcol, ecol, main, adjust_hist=True, extra=('age',), minN=8):
    d = d.copy()
    covs = list(main) + list(extra)
    if adjust_hist:
        vc = d['hist'].value_counts(); d = d[d['hist'].isin(vc[vc >= minN].index)]
        H = hist_dummies(d); d = d.join(H); covs += list(H.columns)
    r = cox(d, tcol, ecol, covs)
    return r.loc[[m for m in main if m in r.index]], r.attrs

def fmt(r, attrs):
    return '; '.join(f"{i}: HR {x.HR:.2f} ({x.lo:.2f}-{x.hi:.2f}) p={x.p:.3g}" for i, x in r.iterrows()) + f"  [n={attrs['n']}, events={attrs['events']}]"

out = []
P = lambda *a: out.append(' '.join(str(x) for x in a))
d = df[df['amp'].notna()]
P('== Cohort ==')
P('n primary w/ CNA:', len(d), ' with RNA:', d['Treg'].notna().sum())
P(pd.crosstab(d['hist'], d['amp']).to_string())
P('MDM2/MDM4 amp n =', int(d['amp'].sum()), '; CDK4 co-amp among amp:', int(((d['amp'] == 1) & (d['cdk4_amp'] == 1)).sum()), '/', int(d['amp'].sum()))

P('\n== 1. Immune scores by MDM2/MDM4 amp (z-score units) ==')
for s in ['Treg', 'Cytotoxic', 'Macrophage', 'Tcell', 'IFNG', 'Cyto_minus_Treg', 'Treg_vs_Tcell']:
    a = d.loc[d.amp == 1, s].dropna(); b = d.loc[d.amp == 0, s].dropna()
    mw = stats.mannwhitneyu(a, b)
    # adjusted for histology via OLS on non-DDLPS + DDLPS dummies
    dd = d[[s, 'amp', 'hist']].dropna(); vc = dd['hist'].value_counts(); dd = dd[dd['hist'].isin(vc[vc >= 8].index)]
    X = np.column_stack([np.ones(len(dd)), dd['amp'], hist_dummies(dd).values]); y = dd[s].values
    beta, res, *_ = np.linalg.lstsq(X, y, rcond=None); resid = y - X @ beta; dof = len(y) - X.shape[1]
    se = np.sqrt(np.sum(resid ** 2) / dof * np.linalg.inv(X.T @ X)[1, 1]); pt = 2 * stats.t.sf(abs(beta[1] / se), dof)
    P(f"{s:16s} amp {a.median():+.2f} vs non-amp {b.median():+.2f}  MWU p={mw.pvalue:.3g} | histology-adjusted diff {beta[1]:+.2f} p={pt:.3g}")

P('\n== 1b. Within non-liposarcoma only (histology not the driver) ==')
nl = d[~d['hist'].str.contains('Liposarcoma', na=False)]
P('non-lipo n =', len(nl), ' amp =', int(nl['amp'].sum()))
for s in ['Treg', 'Cytotoxic', 'Treg_vs_Tcell', 'Cyto_minus_Treg']:
    a = nl.loc[nl.amp == 1, s].dropna(); b = nl.loc[nl.amp == 0, s].dropna()
    if len(a) >= 3: P(f"{s:16s} amp {a.median():+.2f} (n={len(a)}) vs {b.median():+.2f} (n={len(b)}) MWU p={stats.mannwhitneyu(a, b).pvalue:.3g}")

P('\n== 1c. MDM2 expression vs Treg, within DDLPS (all amplified) ==')
dl = d[d['hist'].str.contains('Dedifferentiated', na=False)]
for s in ['Treg', 'Treg_vs_Tcell', 'Cytotoxic']:
    x = dl[['MDM2_expr', s]].dropna(); r = stats.spearmanr(x['MDM2_expr'], x[s]); P(f"DDLPS n={len(x)} MDM2 expr vs {s}: rho={r.statistic:.2f} p={r.pvalue:.3g}")

P('\n== 2. MDM2/MDM4 amp and survival (TCGA, not ICI-treated: prognostic only) ==')
for e in ['OS', 'DSS', 'PFS']:
    r, a = run(d, e + '_t', e + '_e', ['amp'], adjust_hist=False, extra=()); P(f"{e} unadjusted: " + fmt(r, a))
    r, a = run(d, e + '_t', e + '_e', ['amp'], adjust_hist=True); P(f"{e} adj histology+age: " + fmt(r, a))

P('\n== 3. Treg score and survival (per 1 SD) ==')
for e in ['OS', 'DSS', 'PFS']:
    for m in ['Treg', 'Treg_vs_Tcell', 'Cytotoxic', 'Cyto_minus_Treg']:
        dd = d.copy(); dd[m] = (dd[m] - dd[m].mean()) / dd[m].std()
        r0, a0 = run(dd, e + '_t', e + '_e', [m], adjust_hist=False, extra=())
        r1, a1 = run(dd, e + '_t', e + '_e', [m, 'amp'], adjust_hist=True)
        P(f"{e} {m:16s} unadj: {fmt(r0, a0)} || adj hist+age+amp: {fmt(r1.loc[[m]], a1)}")

P('\n== 4. CDK4 co-amplification among MDM2/MDM4-amplified ==')
am = d[d.amp == 1]
P(pd.crosstab(am['hist'], am['cdk4_amp']).to_string())
for s in ['TMB', 'Treg', 'Cytotoxic', 'Treg_vs_Tcell']:
    a = am.loc[am.cdk4_amp == 1, s].dropna(); b = am.loc[am.cdk4_amp == 0, s].dropna()
    if len(b) >= 3: P(f"{s:14s} CDK4+ {a.median():+.2f} (n={len(a)}) vs CDK4- {b.median():+.2f} (n={len(b)}) MWU p={stats.mannwhitneyu(a, b).pvalue:.3g}")
for e in ['OS', 'DSS']:
    try:
        r, a = run(am, e + '_t', e + '_e', ['cdk4_amp'], adjust_hist=False, extra=()); P(f"{e} CDK4 co-amp within amp: " + fmt(r, a))
    except Exception as ex: P(e, 'CDK4 cox failed', ex)

P('\n== 5. TMB ==')
a = d.loc[d.amp == 1, 'TMB'].dropna(); b = d.loc[d.amp == 0, 'TMB'].dropna()
P(f"TMB amp {a.median():.2f} vs {b.median():.2f} MWU p={stats.mannwhitneyu(a, b).pvalue:.3g}")
a = nl.loc[nl.amp == 1, 'TMB'].dropna(); b = nl.loc[nl.amp == 0, 'TMB'].dropna()
if len(a) >= 3: P(f"TMB non-lipo amp {a.median():.2f} (n={len(a)}) vs {b.median():.2f} MWU p={stats.mannwhitneyu(a, b).pvalue:.3g}")

open('results/tcga_sarc_results.txt', 'w').write('\n'.join(out)); print('\n'.join(out))
df.to_csv('results/tcga_sarc_analysis_table.csv')


== Cohort ==
n primary w/ CNA: 253  with RNA: 251
amp                                                                                                  0.0  1.0
hist                                                                                                         
Dedifferentiated Liposarcoma                                                                          16   43
Desmoid/Aggressive Fibromatosis                                                                        2    0
Leiomyosarcoma                                                                                        96    3
Malignant Peripheral Nerve Sheath Tumor                                                                9    0
Myxofibrosarcoma                                                                                      22    2
Synovial Sarcoma                                                                                      10    0
Undifferentiated Pleomorphic Sarcoma/Malignant Fibrous Histiocytoma/Hi

## 5. MSK-IMPACT 50k: TMB and CDK4 co-amplification

In [5]:
"""MSK-IMPACT 50K: TMB and CDK4 co-amplification in MDM2/MDM4-amplified sarcomas."""
import json
import numpy as np
import pandas as pd
from scipy import stats

df = pd.DataFrame(json.load(open("data/msk50k_sarcoma.json")))
for c in ["MDM2", "MDM4", "CDK4"]:
    df[c] = pd.to_numeric(df.get(c), errors="coerce")
df["TMB"] = pd.to_numeric(df["tmb"], errors="coerce")
df["amp"] = ((df.MDM2 == 2) | (df.MDM4 == 2)).astype(int)
df["mdm2_only_def"] = (df.MDM2 == 2).astype(int)
out = []
P = lambda s: (out.append(s), print(s))


def compare(x, label, col="amp"):
    a = x.loc[x[col] == 1, "TMB"].dropna()
    b = x.loc[x[col] == 0, "TMB"].dropna()
    cdk4 = (x.loc[x[col] == 1, "CDK4"] == 2).mean()
    P(f"{label}: n={len(x)}, amplified={len(a)}, median TMB {a.median():.2f} vs {b.median():.2f}, "
      f"Mann-Whitney p={stats.mannwhitneyu(a, b).pvalue:.2e}, CDK4 co-amp {cdk4:.0%}")


main = df[df.cancer_type.isin(["Soft Tissue Sarcoma", "Bone Cancer"])]
compare(main, "STS + bone (main analysis)")
compare(main, "STS + bone, MDM2 amplification only", col="mdm2_only_def")
compare(df, "All sarcoma types incl. uterine")
compare(main.drop_duplicates("patient"), "STS + bone, one sample per patient")
lipo = main["detailed"].fillna("").str.contains("Liposarcoma")
compare(main[~lipo], "STS + bone, non-liposarcoma")
P(f"Share of amplified tumors that are liposarcoma: {lipo[main.amp == 1].mean():.0%}")
P(f"MDM4-only amplified: {((main.MDM4 == 2) & (main.MDM2 != 2)).sum()} of {main.amp.sum()}")

# histology-adjusted: rank-transformed TMB ~ amplification + detailed histology (types with >=10 samples)
x = main[["TMB", "amp", "detailed"]].dropna()
vc = x.detailed.value_counts()
x = x[x.detailed.isin(vc[vc >= 10].index)].copy()
x["r"] = stats.rankdata(x.TMB)
D = pd.get_dummies(x.detailed, drop_first=True, dtype=float)
X = np.column_stack([np.ones(len(x)), x.amp, D.values])
y = x.r.values
b, *_ = np.linalg.lstsq(X, y, rcond=None)
res = y - X @ b
dof = len(y) - X.shape[1]
se = np.sqrt(res @ res / dof * np.linalg.inv(X.T @ X)[1, 1])
P(f"Histology-adjusted (rank TMB ~ amp + detailed histology): coef {b[1]:.1f}, p={2 * stats.t.sf(abs(b[1] / se), dof):.3g}, n={len(x)}")
open("results/msk_impact_tmb_results.txt", "w").write("\n".join(out))


STS + bone (main analysis): n=2897, amplified=442, median TMB 0.88 vs 1.12, Mann-Whitney p=2.88e-16, CDK4 co-amp 86%
STS + bone, MDM2 amplification only: n=2897, amplified=427, median TMB 0.88 vs 1.12, Mann-Whitney p=2.71e-17, CDK4 co-amp 89%
All sarcoma types incl. uterine: n=3293, amplified=462, median TMB 0.88 vs 1.60, Mann-Whitney p=1.74e-19, CDK4 co-amp 84%
STS + bone, one sample per patient: n=2628, amplified=404, median TMB 0.88 vs 1.12, Mann-Whitney p=3.68e-15, CDK4 co-amp 86%
STS + bone, non-liposarcoma: n=2466, amplified=123, median TMB 0.98 vs 1.12, Mann-Whitney p=7.41e-01, CDK4 co-amp 63%
Share of amplified tumors that are liposarcoma: 72%
MDM4-only amplified: 15 of 442
Histology-adjusted (rank TMB ~ amp + detailed histology): coef -238.8, p=0.000767, n=2790


780

## 6. GSE221493 single-cell: MDM2 vs Treg

In [6]:
# GSE221493 (liposarcoma scRNA-seq atlas): does tumor MDM2 expression track Treg abundance?
# Run in Google Colab (fast download from GEO). Needs ~6 GB RAM.

import os, gzip, numpy as np, pandas as pd, scipy.io, scipy.sparse as sp
from scipy import stats

BASE = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE221nnn/GSE221493/suppl/"
FILES = ["GSE221493_sc_LPS_atlas_normalized_counts_barcodes.tsv.gz",
         "GSE221493_sc_LPS_atlas_normalized_counts_features.tsv.gz",
         "GSE221493_sc_LPS_atlas_normalized_counts_matrix.mtx.gz"]
for f in FILES:
    if not os.path.exists(f):
        os.system(f"wget -q {BASE}{f}")

# ---- metadata (column order of the matrix) ----
meta = pd.read_csv(FILES[0], sep="\t", header=None)
meta.columns = ["barcode", "patient", "tissue", "location", "cluster", "lineage", "celltype", "subtype", "umap1", "umap2"][:meta.shape[1]]
meta["sample"] = meta["patient"] + "|" + meta["tissue"]
genes = pd.read_csv(FILES[1], sep="\t", header=None)[0].tolist()

# ---- expression: keep only the genes we need ----
M = scipy.io.mmread(FILES[2]).tocsr()          # genes x cells
want = ["MDM2", "CDK4", "FOXP3"]
expr = pd.DataFrame({g: np.asarray(M[genes.index(g)].todense()).ravel() for g in want})
del M
meta = pd.concat([meta, expr], axis=1)

# ---- per-sample summaries ----
T_SUBS = {"CD4+ T cells", "CD4+ Treg cells", "CD8+ T cells"}
rows = []
for s, d in meta.groupby("sample"):
    tum = d[d["lineage"] == "Tumor cells"]
    rows.append(dict(
        sample=s, patient=d["patient"].iloc[0], tissue=d["tissue"].iloc[0], n_cells=len(d), n_tumor=len(tum),
        MDM2_tumor=tum["MDM2"].mean() if len(tum) >= 20 else np.nan,     # mean MDM2 in tumor cells
        MDM2_all=d["MDM2"].mean(),                                          # mean MDM2 in all cells
        Treg=(d["subtype"] == "CD4+ Treg cells").sum(),
        Tcells=d["subtype"].isin(T_SUBS).sum(),
        CD8=(d["subtype"] == "CD8+ T cells").sum()))
S = pd.DataFrame(rows)
S["Treg_frac_all"] = S["Treg"] / S["n_cells"]
S["Treg_frac_T"] = S["Treg"] / S["Tcells"].where(S["Tcells"] >= 20)
S["CD8_Treg_ratio"] = np.log2((S["CD8"] + 1) / (S["Treg"] + 1))
S["is_tumor"] = ~S["tissue"].isin(["nl", "lipoma"])
S["is_DD"] = S["tissue"].str.startswith("DDLPS-DD")

def corr(df, x, y, label):
    d = df[[x, y]].dropna()
    if len(d) < 5:
        print(f"{label:55s} n={len(d)} (too few)"); return
    p = stats.pearsonr(d[x], d[y]); s = stats.spearmanr(d[x], d[y])
    print(f"{label:55s} n={len(d):2d}  Pearson r={p.statistic:+.2f} (p={p.pvalue:.3g})  Spearman rho={s.statistic:+.2f} (p={s.pvalue:.3g})")

print(S.sort_values("Treg_frac_all").round(3).to_string())
print()
for xm in ["MDM2_all", "MDM2_tumor"]:
    for ym in ["Treg_frac_all", "Treg_frac_T", "CD8_Treg_ratio"]:
        print(f"--- {xm} vs {ym} ---")
        corr(S, xm, ym, "A. all samples (incl. normal fat/lipoma)")
        corr(S[S.is_tumor], xm, ym, "B. tumor samples only")
        P = S[S.is_tumor].groupby("patient")[[xm, ym]].mean()
        corr(P, xm, ym, "C. tumor only, one value per patient")
        corr(S[S.is_tumor & S.is_DD], xm, ym, "D. DD components only")
        corr(S[S.is_tumor & ~S.is_DD], xm, ym, "E. WD components only")
    print()

# within-patient: DD vs paired WD (does the component with higher MDM2 have more Treg?)
pairs = S[S.is_tumor].pivot_table(index="patient", columns="is_DD", values=["MDM2_tumor", "Treg_frac_all", "Treg_frac_T"])
pairs = pairs.dropna()
if len(pairs) >= 4:
    dM = pairs[("MDM2_tumor", True)] - pairs[("MDM2_tumor", False)]
    for y in ["Treg_frac_all", "Treg_frac_T"]:
        dT = pairs[(y, True)] - pairs[(y, False)]
        w = stats.wilcoxon(dT)
        print(f"Paired DD vs WD ({len(pairs)} patients): DD-WD MDM2 median {dM.median():+.3f}; DD-WD {y} median {dT.median():+.4f} (Wilcoxon p={w.pvalue:.3g}); corr of differences rho={stats.spearmanr(dM, dT).statistic:+.2f}")

S.to_csv("GSE221493_sample_level_MDM2_Treg.csv", index=False)


           sample patient     tissue  n_cells  n_tumor  MDM2_tumor  MDM2_all  Treg  Tcells   CD8  Treg_frac_all  Treg_frac_T  CD8_Treg_ratio  is_tumor  is_DD
4          P12|nl     P12         nl      480      262       0.153     0.173     0      42    13          0.000        0.000           3.807     False  False
22    P4|DDLPS-WD      P4   DDLPS-WD     3299     1798       1.393     0.870     0      14     5          0.000          NaN           2.585      True  False
14         P17|nl     P17         nl     2172      906       0.261     0.254     1     119    41          0.000        0.008           4.392     False  False
25          P6|nl      P6         nl     1589      378       0.082     0.115     2     103    38          0.001        0.019           3.700     False  False
20          P3|nl      P3         nl     2486      662       0.436     0.367     4     286   117          0.002        0.014           4.561     False  False
17    P2|DDLPS-DD      P2   DDLPS-DD      801      2